In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROK_API_KEY"] = os.getenv("GROK_API_KEY")

In [2]:
from langchain.document_loaders import TextLoader

In [4]:
loader = TextLoader("C:\code\LLMops_RAG\data\Agentic AI.txt",encoding="utf8")
documents = loader.load()

In [5]:
documents[0].page_content[:500]

'Agentic AI is an advanced form of artificial intelligence that can independently make decisions, plan tasks, and take actions to achieve specific goals with minimal human intervention. Unlike traditional AI systems that only respond to direct instructions, Agentic AI systems can analyze situations, reason through problems, adapt to changing environments, and execute multi-step workflows autonomously.\n\nAn Agentic AI system typically combines several capabilities such as memory, reasoning, plannin'

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [8]:
text_chunks = text_splitter.split_documents(documents)

In [9]:
text_chunks

[Document(metadata={'source': 'C:\\code\\LLMops_RAG\\data\\Agentic AI.txt'}, page_content='Agentic AI is an advanced form of artificial intelligence that can independently make decisions, plan tasks, and take actions to achieve specific goals with minimal human intervention. Unlike'),
 Document(metadata={'source': 'C:\\code\\LLMops_RAG\\data\\Agentic AI.txt'}, page_content='Unlike traditional AI systems that only respond to direct instructions, Agentic AI systems can analyze situations, reason through problems, adapt to changing environments, and execute multi-step'),
 Document(metadata={'source': 'C:\\code\\LLMops_RAG\\data\\Agentic AI.txt'}, page_content='execute multi-step workflows autonomously.'),
 Document(metadata={'source': 'C:\\code\\LLMops_RAG\\data\\Agentic AI.txt'}, page_content='An Agentic AI system typically combines several capabilities such as memory, reasoning, planning, tool usage, and continuous learning. These systems can interact with external tools, APIs, database

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\code\LLMops_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1546.73it/s]


In [17]:
vectore_store = FAISS.from_documents(text_chunks,embeddings)

In [18]:
vectore_store

In [19]:
retriever = vectore_store.as_retriever()

In [20]:
query = "What is Agentic AI?"
docs = vectore_store.similarity_search(query,k=4)

for i , doc in enumerate(docs):
    print(f'Document {i+1}:')
    print(doc.page_content)
    print("-"*50)

Document 1:
Agentic AI is an advanced form of artificial intelligence that can independently make decisions, plan tasks, and take actions to achieve specific goals with minimal human intervention. Unlike
--------------------------------------------------
Document 2:
Key characteristics of Agentic AI include:
--------------------------------------------------
Document 3:
Applications of Agentic AI include:
--------------------------------------------------
Document 4:
Agentic AI represents the next evolution of intelligent systems by enabling machines to act more like autonomous digital assistants capable of solving complex real-world problems.
--------------------------------------------------


In [21]:
from langchain.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [22]:
prompt = ChatPromptTemplate.from_template(template)

In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [24]:
from langchain.schema.output_parser import StrOutputParser

In [25]:
output_parser = StrOutputParser()

In [30]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile",
            temperature=0,
            api_key= os.getenv("GROK_API_KEY")
            )

In [31]:
from langchain.schema.runnable import RunnablePassthrough

rag_chain = (
    {"context":retriever , "question":RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [32]:
rag_chain.invoke("tell me about Agentic AI")

"Agentic AI is an advanced form of artificial intelligence that can independently make decisions and take actions with minimal human intervention. It can plan tasks and achieve specific goals. Agentic AI enables machines to act like autonomous digital assistants, solving complex real-world problems. Key characteristics and applications of Agentic AI are mentioned in the context, but not fully described. Agentic AI represents the next evolution of intelligent systems. It can make decisions, plan tasks, and take actions autonomously. The context does not provide a full list of its characteristics and applications. Agentic AI is capable of solving complex problems. I don't know more details about its specific applications and characteristics."